In [ ]:
# ============================================================
# KG1 v51 DEFINITIVE — SINGLE CELL
# Fixes ALL errors: stubs, dtype, OOM, versions, download
# ============================================================

!pip install -q peft datasets accelerate trl huggingface_hub safetensors pandas

import subprocess, sys, os, json, random, time, zipfile, shutil, re, math, types, gc
import importlib, importlib.machinery
from datetime import datetime, timezone
from collections import Counter

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ==================== STUBS ====================
class _Stub:
    def __init__(self, *a, **kw): pass
    def __call__(self, *a, **kw): return a[0] if a else None
    def __getattr__(self, name): return _Stub()

for pkg in ['mamba_ssm', 'mamba_ssm.ops', 'mamba_ssm.ops.triton',
            'mamba_ssm.ops.triton.layernorm_gated',
            'mamba_ssm.ops.triton.selective_state_update',
            'mamba_ssm.ops.triton.ssd_combined',
            'mamba_ssm.utils', 'mamba_ssm.utils.generation',
            'causal_conv1d', 'causal_conv1d.causal_conv1d_interface']:
    if pkg not in sys.modules:
        m = types.ModuleType(pkg)
        m.__version__ = '0.0.0'
        m.__spec__ = importlib.machinery.ModuleSpec(pkg, None)
        m.__path__ = []
        m.__file__ = 'stub'
        for attr in ['RMSNormGated', 'rmsnorm_fn', 'selective_state_update',
                     'mamba_chunk_scan_combined', 'mamba_split_conv1d_scan_combined',
                     'InferenceParams', 'GenerationMixin',
                     'causal_conv1d_fn', 'causal_conv1d_update']:
            setattr(m, attr, _Stub)
        sys.modules[pkg] = m

ms = sys.modules['mamba_ssm']
ms.ops = sys.modules['mamba_ssm.ops']
ms.ops.triton = sys.modules['mamba_ssm.ops.triton']
ms.ops.triton.layernorm_gated = sys.modules['mamba_ssm.ops.triton.layernorm_gated']
ms.ops.triton.selective_state_update = sys.modules['mamba_ssm.ops.triton.selective_state_update']
ms.ops.triton.ssd_combined = sys.modules['mamba_ssm.ops.triton.ssd_combined']
ms.utils = sys.modules['mamba_ssm.utils']
ms.utils.generation = sys.modules['mamba_ssm.utils.generation']
cc = sys.modules['causal_conv1d']
cc.causal_conv1d_fn = _Stub()
cc.causal_conv1d_update = _Stub()
print('Stubs OK')

import torch

# ==================== FIX DTYPE BUG ====================
# PyTorch 2.10+ rejects dtype as input to F.linear.
# PyTorch 2.8 (where v50c worked) silently handled it.
# This patch restores 2.8 behavior for non-tensor inputs.
_orig_linear = torch.nn.functional.linear
def _safe_linear(input, weight, bias=None):
    if not isinstance(input, torch.Tensor):
        input = torch.zeros(1, weight.shape[-1], dtype=weight.dtype, device=weight.device)
    return _orig_linear(input, weight, bias)
torch.nn.functional.linear = _safe_linear
print('F.linear dtype patch OK')

print(f'PyTorch: {torch.__version__}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'VRAM: {vram_gb:.1f} GB')

try:
    from transformers.utils.import_utils import is_flash_attn_greater_or_equal_2_10
except ImportError:
    import transformers.utils.import_utils as _tiu
    _tiu.is_flash_attn_greater_or_equal_2_10 = lambda: False

import pandas as pd
from huggingface_hub import HfApi, login, hf_hub_download

# ==================== AUTH ====================
def _get_secret(*names):
    try:
        from google.colab import userdata
        for n in names:
            v = userdata.get(n)
            if v: return v
    except Exception: pass
    for n in names:
        v = os.environ.get(n)
        if v: return v
    return ''

HF_TOKEN = _get_secret('HF_KEY', 'HF_TOKEN')
if HF_TOKEN:
    login(token=HF_TOKEN)
    os.environ['HF_TOKEN'] = HF_TOKEN
    print('HF login OK')

KAGGLE_USERNAME = _get_secret('KAGGLE_USERNAME') or 'felipe1983'
KAGGLE_KEY = _get_secret('KAGGLE_KEY')
if KAGGLE_KEY:
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    kpath = os.path.expanduser('~/.kaggle/kaggle.json')
    with open(kpath, 'w') as f:
        json.dump({'username': KAGGLE_USERNAME, 'key': KAGGLE_KEY}, f)
    os.chmod(kpath, 0o600)
    print(f'Kaggle: {KAGGLE_USERNAME}')

api = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()

# ==================== CONFIG ====================
DATA_REPO = 'felipesp1983/kg1-nemotron-training'
OUTPUT_REPO = 'felipesp1983/kg1-nemotron-lora-v51-perfect'
MODEL_NAME = 'nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16'
MODEL_REVISION = '378df16e4b54'
COMPETITION = 'nvidia-nemotron-model-reasoning-challenge'
N_EXAMPLES = 5000
N_EPOCHS = 2
SUBMIT_STEPS = [200, 400, 600, 800, 1000, 1200]
CONFIG = {
    'lora_rank': 32, 'lora_alpha': 16, 'lora_dropout': 0.05,
    'target_modules': 'all-linear', 'learning_rate': 5e-5,
    'per_device_batch_size': 1, 'gradient_accumulation_steps': 8,
    'max_length': 1024, 'warmup_ratio': 0.05, 'weight_decay': 0.01,
    'lr_scheduler': 'cosine', 'optim': 'adamw_torch',
    'output_dir': '/tmp/kg1_output/v51',
}

try:
    for p in ['/usr/local/cuda-12.8/bin/ptxas', '/usr/local/cuda/bin/ptxas']:
        if os.path.exists(p):
            t = os.path.join(os.path.dirname(shutil.which('python') or '/usr/bin/python'), 'ptxas')
            if not os.path.exists(t): shutil.copy2(p, t)
            break
except Exception: pass

print(f'Config: {N_EXAMPLES}ex, {N_EPOCHS}ep, r={CONFIG["lora_rank"]}, a={CONFIG["lora_alpha"]}')

# ==================== LOAD DATA ====================
print('\n=== Loading data ===')
all_examples = []
try:
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/sft_v51_perfect.jsonl', local_dir='/tmp/kg1_data')
    with open('/tmp/kg1_data/data/sft_v51_perfect.jsonl') as f:
        for line in f:
            all_examples.append(json.loads(line))
    print(f'Loaded: {len(all_examples)} examples')
except Exception as e:
    print(f'Failed ({e}), using train.csv')
    hf_hub_download(repo_id=DATA_REPO, repo_type='dataset',
                    filename='data/train.csv', local_dir='/tmp/kg1_data')
    for _, row in pd.read_csv('/tmp/kg1_data/data/train.csv').iterrows():
        all_examples.append({
            'prompt': row['prompt'] + '\nPut your final answer inside \\boxed{}.',
            'completion': '\\boxed{' + str(row['answer']) + '}',
            'family': 'unknown'})

def classify(t):
    p = t.lower()
    if 'bit manipulation' in p: return 'bit'
    if 'gravitational' in p: return 'grav'
    if 'unit conversion' in p or 'measurement' in p: return 'unit'
    if 'numeral' in p: return 'num'
    if 'encryption' in p: return 'enc'
    if 'transformation' in p: return 'eq'
    return 'other'

random.seed(42)
by_family = {}
for ex in all_examples:
    fam = ex.get('family') or classify(ex.get('prompt', ''))
    by_family.setdefault(fam, []).append(ex)

shares = {'grav':1,'unit':1,'num':1,'enc':1,'cipher':1,'bit':1.5,'eq':2.5,'equation':2.5,'gravity':1,'numeral':1}
total_shares = sum(shares.get(f, 1.0) for f in by_family)
base_n = N_EXAMPLES / total_shares
examples = []
for fam, pool in by_family.items():
    n = int(base_n * shares.get(fam, 1.0))
    if n <= len(pool): examples.extend(random.sample(pool, n))
    else: examples.extend(pool + random.choices(pool, k=n - len(pool)))
random.shuffle(examples)
examples = examples[:N_EXAMPLES]

formatted = [{'messages': [
    {'role': 'user', 'content': ex.get('prompt', '')},
    {'role': 'assistant', 'content': ex.get('completion', '')}]} for ex in examples]

fam_counts = Counter(classify(e['messages'][0]['content']) for e in formatted)
print(f'Dataset: {len(formatted)} examples')
for fam, cnt in sorted(fam_counts.items()):
    print(f'  {fam}: {cnt}')

# ==================== LOAD MODEL ====================
print(f'\n=== Loading model (revision {MODEL_REVISION}) ===')
gc.collect()
torch.cuda.empty_cache()

from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    revision=MODEL_REVISION,
    device_map={'': 0},
    trust_remote_code=True,
    torch_dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
)

for module in model.modules():
    if hasattr(module, 'is_fast_path_available'):
        module.is_fast_path_available = False

print(f'Model: {model.num_parameters()/1e9:.1f}B, VRAM: {torch.cuda.memory_allocated()/1e9:.1f}GB')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Test forward pass
print('Testing forward pass...')
try:
    dummy = tokenizer('Hello', return_tensors='pt').to('cuda')
    with torch.no_grad():
        model(**dummy)
    print('Forward pass OK!')
except Exception as e:
    print(f'Forward pass issue: {e} (training may still work)')

# ==================== LORA ====================
print(f'\n=== LoRA r={CONFIG["lora_rank"]} a={CONFIG["lora_alpha"]} ===')
model.enable_input_require_grads()
model = get_peft_model(model, LoraConfig(
    r=CONFIG['lora_rank'], lora_alpha=CONFIG['lora_alpha'],
    lora_dropout=CONFIG['lora_dropout'], target_modules=CONFIG['target_modules'],
    bias='none', task_type='CAUSAL_LM'))
model.print_trainable_parameters()

# ==================== DATASET ====================
from datasets import Dataset
texts = [tokenizer.apply_chat_template(ex['messages'], tokenize=False, add_generation_prompt=False) for ex in formatted]
ds = Dataset.from_dict({'text': texts})
print(f'{len(ds)} examples')

# ==================== TRAIN ====================
print('\n=== Training ===')
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback
os.makedirs(CONFIG['output_dir'], exist_ok=True)

# Detect SFTConfig API (max_length vs max_seq_length)
import inspect
sft_params = inspect.signature(SFTConfig).parameters
length_key = 'max_seq_length' if 'max_seq_length' in sft_params else 'max_length'
print(f'SFTConfig length param: {length_key}')

sft_kwargs = {
    'output_dir': CONFIG['output_dir'],
    'dataset_text_field': 'text',
    length_key: CONFIG['max_length'],
    'packing': False,
    'num_train_epochs': N_EPOCHS,
    'per_device_train_batch_size': CONFIG['per_device_batch_size'],
    'gradient_accumulation_steps': CONFIG['gradient_accumulation_steps'],
    'learning_rate': CONFIG['learning_rate'],
    'warmup_ratio': CONFIG['warmup_ratio'],
    'weight_decay': CONFIG['weight_decay'],
    'lr_scheduler_type': CONFIG['lr_scheduler'],
    'optim': CONFIG['optim'],
    'bf16': True,
    'logging_steps': 5,
    'save_strategy': 'steps',
    'save_steps': 100,
    'save_total_limit': 15,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'report_to': 'none',
    'dataloader_num_workers': 0,
    'max_grad_norm': 1.0,
}
training_args = SFTConfig(**sft_kwargs)

class UploadCB(TrainerCallback):
    def __init__(self, repo):
        self.repo = repo
        self.hf = HfApi(token=HF_TOKEN) if HF_TOKEN else HfApi()
        try: self.hf.create_repo(repo, private=True, exist_ok=True)
        except: pass
    def on_save(self, args, state, control, **kw):
        import glob as g
        step = state.global_step
        loss = 'N/A'
        if state.log_history:
            for e in reversed(state.log_history):
                if 'loss' in e: loss = e['loss']; break
        ckpts = sorted(g.glob(f'{args.output_dir}/checkpoint-*'))
        if not ckpts: return
        try:
            self.hf.upload_folder(folder_path=ckpts[-1], path_in_repo=f'checkpoint-{step}',
                repo_id=self.repo, commit_message=f'Step {step} Loss {loss}')
            print(f'\n>>> HF OK: step {step}, loss={loss}')
        except Exception as e:
            print(f'\n>>> HF FAIL: {e}')
    def on_log(self, args, state, control, logs=None, **kw):
        if not logs: return
        loss = logs.get('loss', 0)
        if isinstance(loss, float) and (math.isnan(loss) or math.isinf(loss)):
            print(f'\n!!! NaN at step {state.global_step}')
            control.should_training_stop = True

# Detect SFTTrainer API (tokenizer vs processing_class)
sft_trainer_params = inspect.signature(SFTTrainer).parameters
tok_key = 'processing_class' if 'processing_class' in sft_trainer_params else 'tokenizer'
print(f'SFTTrainer tokenizer param: {tok_key}')

trainer = SFTTrainer(
    model=model,
    train_dataset=ds,
    **{tok_key: tokenizer},
    args=training_args,
    callbacks=[UploadCB(OUTPUT_REPO)],
)

total_steps = (len(ds) // CONFIG['gradient_accumulation_steps']) * N_EPOCHS
print(f'Steps: ~{total_steps}')

start = time.time()
try:
    trainer.train()
except Exception as e:
    print(f'\n!!! Error: {e}')
    model.save_pretrained(CONFIG['output_dir'])
    tokenizer.save_pretrained(CONFIG['output_dir'])

elapsed = time.time() - start
print(f'\nDone: {elapsed/3600:.2f}h')

# ==================== SAVE ====================
model.save_pretrained(CONFIG['output_dir'])
tokenizer.save_pretrained(CONFIG['output_dir'])
final_loss = 'N/A'
if trainer.state.log_history:
    for e in reversed(trainer.state.log_history):
        if 'loss' in e: final_loss = e['loss']; break
try:
    api.create_repo(OUTPUT_REPO, private=True, exist_ok=True)
    api.upload_folder(folder_path=CONFIG['output_dir'], repo_id=OUTPUT_REPO,
        commit_message=f'v51: {len(formatted)}ex loss={final_loss}')
except: pass

# ==================== SMART STRIP + SUBMIT ====================
print('\n=== Smart Strip ===')
import glob
from safetensors.torch import load_file, save_file
ckpt_dir = f'{CONFIG["output_dir"]}/checkpoint-400'
if not os.path.exists(ckpt_dir):
    ckpts = sorted(glob.glob(f'{CONFIG["output_dir"]}/checkpoint-*'),
                   key=lambda x: int(x.split('-')[-1]))
    ckpt_dir = ckpts[-1] if ckpts else CONFIG['output_dir']
tensors = load_file(os.path.join(ckpt_dir, 'adapter_model.safetensors'))
routed_re = re.compile(r'\.experts\.\d+\.')
keep = {k: v for k, v in tensors.items() if not routed_re.search(k)}
print(f'Kept: {len(keep)} | Removed: {len(tensors)-len(keep)}')
out_dir = '/tmp/kg1_submit/stripped'
os.makedirs(out_dir, exist_ok=True)
save_file(keep, os.path.join(out_dir, 'adapter_model.safetensors'))
with open(os.path.join(ckpt_dir, 'adapter_config.json')) as f: cfg = json.load(f)
mods = set()
for k in keep:
    for m in ['q_proj','k_proj','v_proj','o_proj','in_proj','out_proj','up_proj','down_proj','gate']:
        if m in k: mods.add(m)
cfg['target_modules'] = sorted(mods)
with open(os.path.join(out_dir, 'adapter_config.json'), 'w') as f:
    json.dump(cfg, f, indent=2)
zip_path = '/tmp/kg1_submit/submission.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fn in ['adapter_config.json', 'adapter_model.safetensors']:
        zf.write(os.path.join(out_dir, fn), fn)
print(f'ZIP: {os.path.getsize(zip_path)/1e6:.1f} MB')
step_str = ckpt_dir.split('-')[-1] if 'checkpoint' in ckpt_dir else 'final'
desc = f'v51-step{step_str}-smart-strip'
os.system(f'kaggle competitions submit -c {COMPETITION} -f {zip_path} -m "{desc}"')
print(f'\nDONE: {desc} | Loss: {final_loss} | {elapsed/3600:.2f}h')
